In [1]:
import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from sklearn.preprocessing import OneHotEncoder

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [5]:
from xgboost import XGBRegressor

In [6]:
import joblib

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

In [8]:
import warnings
warnings.filterwarnings(action = "ignore")

In [9]:
car_df = pd.read_csv("cardekho_dataset.csv")

In [10]:
car_df.drop(["Unnamed: 0", "model"], axis = 1, inplace = True)

In [11]:
car_df.head()

,car_name,brand,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [12]:
car_df.isna().sum()

car_name             0
brand                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [13]:
cleaned_car_df = car_df[(car_df["km_driven"] < 500000) & (car_df["vehicle_age"] <= 15)]

In [18]:
def change_fuel_type(x):
    print(X)
    return
    return X.replace("Electric", "Hybrid")


trf = ColumnTransformer([
    ("fuel_replace", FunctionTransformer(change_fuel_type), ["fuel_type"])
], remainder="passthrough")

In [57]:
cleaned_car_df.loc[cleaned_car_df["car_name"] == "Nissan Kicks", "seats"] = 5

In [58]:
cleaned_car_df["fuel_type"].replace("Electric", "Hybrid", inplace=True)

In [59]:
cleaned_car_df.loc[:, "selling_price_in_lakhs"] = cleaned_car_df["selling_price"] / 100000

In [60]:
cleaned_car_df.drop(["selling_price"], axis = 1, inplace = True)

In [61]:
cleaned_car_df["vehicle_age"] = cleaned_car_df["vehicle_age"].replace(0, 1)

In [62]:
cleaned_car_df["km_driven_per_year"] = round(cleaned_car_df["km_driven"] / cleaned_car_df["vehicle_age"], 0)

In [63]:
cleaned_car_df["power_to_engine_ratio"] = round(cleaned_car_df["max_power"] * 1000 / cleaned_car_df["engine"], 2) 

In [64]:
log_cols = ["km_driven", "engine", "max_power", "km_driven_per_year", "selling_price_in_lakhs"]

for col in log_cols:
    cleaned_car_df[col] = np.log1p(cleaned_car_df[col]).round(4)

In [65]:
cleaned_car_df["seats"] = cleaned_car_df["seats"].astype(str)

In [66]:
cleaned_car_df.columns

Index(['car_name', 'brand', 'vehicle_age', 'km_driven', 'seller_type',
       'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power',
       'seats', 'selling_price_in_lakhs', 'km_driven_per_year',
       'power_to_engine_ratio'],
      dtype='object')

In [67]:
categorical_columns = cleaned_car_df.select_dtypes(include=['object']).columns.tolist()
numeric_columns = cleaned_car_df.drop(columns=categorical_columns + ["selling_price_in_lakhs"]).columns.tolist()

In [68]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
    ],
    remainder="passthrough"
)

In [70]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(colsample_bytree = 0.8, gamma = 0, learning_rate = 0.05, max_depth = 7, n_estimators = 300, subsample = 0.7))
    ]
)

In [71]:
X = cleaned_car_df.drop("selling_price_in_lakhs", axis=1)
y = cleaned_car_df["selling_price_in_lakhs"]

pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['car_name', 'brand',
                                                   'seller_type', 'fuel_type',
                                                   'transmission_type',
                                                   'seats'])])),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=0.8, devic...
                              feature_types=None, feature_weights=None, gamma=0,
                              grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=7, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=300, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [72]:
joblib.dump(pipeline, "car_price_pipeline.pkl")

['car_price_pipeline.pkl']

In [74]:
import sklearn
print(sklearn.__version__)

1.6.1


In [98]:
input_df

,car_name,brand,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,km_driven_per_year,power_to_engine_ratio
0,Maruti Swift Dzire,Maruti,7,10.8376,Dealer,Diesel,Manual,19.3,7.1301,4.3162,5,8.8918,59.21


In [ ]:
pipeline.predict()

In [22]:
categorical_columns = cleaned_car_df.select_dtypes(include=['object']).columns.tolist()

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

one_hot_encoded = encoder.fit_transform(cleaned_car_df[categorical_columns])

one_hot_df = pd.DataFrame(
    one_hot_encoded,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=cleaned_car_df.index
)

df_encoded = pd.concat([cleaned_car_df.drop(columns=categorical_columns), one_hot_df], axis=1)

In [25]:
X = df_encoded.drop("selling_price_in_lakhs", axis=1)
y = df_encoded["selling_price_in_lakhs"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [26]:
best_xgb = XGBRegressor(colsample_bytree = 0.8, gamma = 0, learning_rate = 0.05, max_depth = 7, n_estimators = 300, subsample = 0.7)

best_xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [36]:
y_pred = best_xgb.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred)).round(2)
r2 = round(r2_score(y_test, y_pred), 4)

In [37]:
f"The XGBoost model has RMSE value of {rmse} and R-2 squared value of {r2}."

'The XGBoost model has RMSE value of 0.13 and R-2 squared value of 0.9485.'

In [ ]:
X_test